# Moirai-2 Training v1 - Multi-country Electricity Forecasting

Muc tieu: pretrain mo hinh Moirai-2 style tren du lieu da quoc gia, luu checkpoint de transfer learning sang Viet Nam.

In [7]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Torch: {torch.__version__}')
print(f'Device: {DEVICE}')

BASE_DIR = Path('.')
DATA_PATH = BASE_DIR / 'data' / 'processed' / 'training_data' / 'tft_premodel_dataset_EDA.csv'
CKPT_DIR = BASE_DIR / 'checkpoint'
LOG_DIR = BASE_DIR / 'lightning_logs' / 'moirai2_v1'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Fair benchmark settings aligned with TFT pretrain policy where possible.
CFG = dict(
    seed=42,
    context_length=24,
    prediction_length=6,
    min_series_length=30,
    val_cutoff_months=12,
    d_model=192,
    nhead=6,
    num_layers=4,
    ff_mult=4,
    dropout=0.15,
    quantiles=[0.1, 0.25, 0.5, 0.75, 0.9],
    learning_rate=3e-4,
    weight_decay=1e-4,
    batch_size=64,
    max_epochs=80,
    patience=12,
    gradient_clip_val=0.1,
    num_workers=0,
    mixed_precision=False,
 )

pl.seed_everything(CFG['seed'], workers=True)
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])

print('Config:')
for k, v in CFG.items():
    print(f'  {k:<24}: {v}')

Seed set to 42


Torch: 2.5.1+cu121
Device: cuda
Config:
  seed                    : 42
  context_length          : 24
  prediction_length       : 6
  min_series_length       : 30
  val_cutoff_months       : 12
  d_model                 : 192
  nhead                   : 6
  num_layers              : 4
  ff_mult                 : 4
  dropout                 : 0.15
  quantiles               : [0.1, 0.25, 0.5, 0.75, 0.9]
  learning_rate           : 0.0003
  weight_decay            : 0.0001
  batch_size              : 64
  max_epochs              : 80
  patience                : 12
  gradient_clip_val       : 0.1
  num_workers             : 0
  mixed_precision         : False


In [8]:
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['entity', 'series', 'date']).reset_index(drop=True)

target_series = [
    'Coal', 'Gas', 'Hydro', 'Solar', 'Wind',
    'Bioenergy', 'Nuclear', 'Other Fossil', 'Other Renewables'
 ]
df = df[df['series'].isin(target_series)].copy()

weather_cols = ['temperature', 'solar', 'humidity', 'precipitation']
for col in weather_cols:
    if col in df.columns:
        df[col] = df.groupby(['entity', 'series'])[col].shift(1)

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    if col != 'generation_TWh':
        df[col] = df.groupby(['entity', 'series'])[col].transform(lambda x: x.interpolate().bfill().ffill())

counts = df.groupby(['entity', 'series']).size()
valid_groups = counts[counts >= CFG['min_series_length']].index
df = df.set_index(['entity', 'series']).loc[valid_groups].reset_index()

df['time_idx'] = df.groupby(['entity', 'series'])['date'].rank(method='dense').astype(int) - 1

df = df.replace([np.inf, -np.inf], np.nan)
df = df.groupby(['entity', 'series'], as_index=False, group_keys=False).apply(lambda g: g.ffill().bfill())

training_cutoff = int(df['time_idx'].max()) - CFG['val_cutoff_months']

print(f'Shape: {df.shape}')
print(f"Groups: {df.groupby(['entity', 'series']).ngroups}")
print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}")
print(f'Training cutoff time_idx: {training_cutoff}')

Shape: (13859, 30)
Groups: 151
Date range: 2018-01-01 -> 2025-12-01
Training cutoff time_idx: 83


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_1808\2128985177.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(['entity', 'series'], as_index=False, group_keys=False).apply(lambda g: g.ffill().bfill())


In [9]:
KNOWN_REAL_CANDIDATES = ['time_idx', 'month', 'month_sin', 'month_cos', 'quarter', 'year']
UNKNOWN_REAL_CANDIDATES = [
    'temperature', 'solar', 'humidity', 'precipitation',
    'gen_lag_1', 'gen_lag_12', 'temp_anomaly', 'solar_norm',
    'is_dry_season', 'is_flood_season', 'quarter_sin', 'quarter_cos',
    'IPI_Value', 'CPI_Value', 'GDP_trillion', 'Oil_Price',
    'FDI_disbursed', 'gas_price', 'castlecoal_price'
]

feature_cols = [c for c in (KNOWN_REAL_CANDIDATES + UNKNOWN_REAL_CANDIDATES) if c in df.columns]
target_col = 'generation_TWh'
group_cols = ['entity', 'series']

print('Feature cols used:', feature_cols)
print('Num features:', len(feature_cols))

Feature cols used: ['time_idx', 'month', 'month_sin', 'month_cos', 'quarter', 'year', 'temperature', 'solar', 'humidity', 'precipitation', 'gen_lag_1', 'gen_lag_12', 'temp_anomaly', 'solar_norm']
Num features: 14


In [10]:
class WindowDataset(Dataset):
    def __init__(self, data, group_cols, feature_cols, target_col, context_length, prediction_length, cutoff_idx=None, split='train'):
        self.samples = []
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.context_length = context_length
        self.prediction_length = prediction_length

        grouped = data.groupby(group_cols)
        for _, g in grouped:
            g = g.sort_values('time_idx').reset_index(drop=True)
            x_feat = g[feature_cols].values.astype(np.float32)
            y_val = g[target_col].values.astype(np.float32)
            t_idx = g['time_idx'].values.astype(int)

            max_start = len(g) - context_length - prediction_length
            if max_start < 0:
                continue

            for start in range(max_start + 1):
                enc_start = start
                enc_end = start + context_length
                dec_end = enc_end + prediction_length

                last_encoder_idx = t_idx[enc_end - 1]
                if cutoff_idx is not None:
                    if split == 'train' and last_encoder_idx > cutoff_idx:
                        continue
                    if split == 'val' and last_encoder_idx <= cutoff_idx:
                        continue

                x_hist_feat = x_feat[enc_start:enc_end]
                x_hist_target = y_val[enc_start:enc_end].reshape(-1, 1)
                x_hist = np.concatenate([x_hist_target, x_hist_feat], axis=1)
                y_future = y_val[enc_end:dec_end]

                self.samples.append((x_hist, y_future))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x_hist, y_future = self.samples[idx]
        return torch.from_numpy(x_hist), torch.from_numpy(y_future)

train_ds = WindowDataset(
    data=df,
    group_cols=group_cols,
    feature_cols=feature_cols,
    target_col=target_col,
    context_length=CFG['context_length'],
    prediction_length=CFG['prediction_length'],
    cutoff_idx=training_cutoff,
    split='train'
)

val_ds = WindowDataset(
    data=df,
    group_cols=group_cols,
    feature_cols=feature_cols,
    target_col=target_col,
    context_length=CFG['context_length'],
    prediction_length=CFG['prediction_length'],
    cutoff_idx=training_cutoff,
    split='val'
)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG['batch_size'],
    shuffle=True,
    num_workers=CFG['num_workers']
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG['batch_size'] * 2,
    shuffle=False,
    num_workers=CFG['num_workers']
)

x0, y0 = next(iter(train_loader))
print(f'Train samples: {len(train_ds):,}')
print(f'Val samples: {len(val_ds):,}')
print(f'Input shape: {x0.shape}  (B, context_length, 1+num_features)')
print(f'Target shape: {y0.shape} (B, prediction_length)')

Train samples: 8,848
Val samples: 632
Input shape: torch.Size([64, 24, 15])  (B, context_length, 1+num_features)
Target shape: torch.Size([64, 6]) (B, prediction_length)


In [11]:
class Moirai2Forecaster(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, ff_mult, dropout, prediction_length, quantiles):
        super().__init__()
        self.prediction_length = prediction_length
        self.quantiles = quantiles
        self.nq = len(quantiles)

        self.input_proj = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * ff_mult,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, prediction_length * self.nq)

    def forward(self, x):
        z = self.input_proj(x)
        z = self.encoder(z)
        z = self.norm(z[:, -1, :])
        out = self.head(z)
        return out.view(-1, self.prediction_length, self.nq)

def quantile_loss(pred_q, target, quantiles):
    losses = []
    for i, q in enumerate(quantiles):
        err = target - pred_q[:, :, i]
        losses.append(torch.maximum((q - 1) * err, q * err).unsqueeze(-1))
    return torch.cat(losses, dim=-1).mean()

def compute_metrics(y_true, y_pred):
    mae = torch.mean(torch.abs(y_true - y_pred)).item()
    rmse = torch.sqrt(torch.mean((y_true - y_pred) ** 2)).item()
    smape = (200.0 * torch.mean(torch.abs(y_true - y_pred) / (torch.abs(y_true) + torch.abs(y_pred) + 1e-8))).item()
    wape = (100.0 * torch.sum(torch.abs(y_true - y_pred)) / (torch.sum(torch.abs(y_true)) + 1e-8)).item()
    ss_res = torch.sum((y_true - y_pred) ** 2)
    ss_tot = torch.sum((y_true - torch.mean(y_true)) ** 2)
    r2 = (1 - ss_res / (ss_tot + 1e-8)).item()
    return mae, rmse, smape, wape, r2

class LitMoirai2(pl.LightningModule):
    def __init__(self, cfg, input_dim):
        super().__init__()
        self.save_hyperparameters({'cfg': cfg, 'input_dim': input_dim})
        self.cfg = cfg
        self.model = Moirai2Forecaster(
            input_dim=input_dim,
            d_model=cfg['d_model'],
            nhead=cfg['nhead'],
            num_layers=cfg['num_layers'],
            ff_mult=cfg['ff_mult'],
            dropout=cfg['dropout'],
            prediction_length=cfg['prediction_length'],
            quantiles=cfg['quantiles']
        )

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        pred_q = self(x)
        loss = quantile_loss(pred_q, y, self.cfg['quantiles'])
        self.log('train_loss', loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        pred_q = self(x)
        loss = quantile_loss(pred_q, y, self.cfg['quantiles'])
        median_idx = self.cfg['quantiles'].index(0.5) if 0.5 in self.cfg['quantiles'] else len(self.cfg['quantiles']) // 2
        y_hat = pred_q[:, :, median_idx]
        mae, rmse, smape, wape, r2 = compute_metrics(y, y_hat)
        self.log('val_loss', loss, prog_bar=True, on_epoch=True, on_step=False)
        self.log('val_mae', mae, prog_bar=False)
        self.log('val_rmse', rmse, prog_bar=False)
        self.log('val_smape', smape, prog_bar=False)
        self.log('val_wape', wape, prog_bar=False)
        self.log('val_r2', r2, prog_bar=False)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.cfg['learning_rate'],
            weight_decay=self.cfg['weight_decay']
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=2,
            min_lr=1e-6
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss'
            }
        }

def freeze_encoder_for_phase1(model):
    frozen = 0
    total = 0
    for name, p in model.named_parameters():
        total += p.numel()
        # TFT phase-1 keeps heads/trainable components adapting; mimic that behavior.
        if name.startswith('encoder'):
            p.requires_grad = False
            frozen += p.numel()
        else:
            p.requires_grad = True
    return frozen, total

def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True

In [12]:
input_dim = 1 + len(feature_cols)
lit_model = LitMoirai2(CFG, input_dim=input_dim)

checkpoint_callback = ModelCheckpoint(
    dirpath=str(CKPT_DIR),
    filename='moirai2_v1_{epoch:02d}_{val_loss:.4f}',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    save_last=False,
    verbose=True
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=CFG['patience'],
    mode='min',
    verbose=True
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')
logger = TensorBoardLogger(save_dir=str(BASE_DIR / 'lightning_logs'), name='moirai2_v1')

trainer = pl.Trainer(
    accelerator='gpu' if DEVICE == 'cuda' else 'cpu',
    devices=1,
    max_epochs=CFG['max_epochs'],
    gradient_clip_val=CFG['gradient_clip_val'],
    callbacks=[checkpoint_callback, early_stop, lr_monitor],
    logger=logger,
    enable_progress_bar=True,
    log_every_n_steps=10,
    precision='16-mixed' if (DEVICE == 'cuda' and CFG.get('mixed_precision', False)) else '32'
)

trainer.fit(lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

print('Training done')
print(f'Best val_loss: {checkpoint_callback.best_model_score}')
print(f'Best ckpt: {checkpoint_callback.best_model_path}')

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 3050 Ti Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting\checkpoint exists and is not empty.
LOCAL_RANK: 0 - CUDA_V

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Moirai2Forecaster │  1.8 M │ train │     0 │
└───┴───────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 1.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.8 M                                                                                                
Total estimated model params size (MB): 7                                                                          
Modules in train mode: 46                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightning\pytorch\trainer\connectors\data_
connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing 
the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

...

Metric val_loss improved by 0.000 >= min_delta = 0.0. New best score: 0.240
Epoch 76, global step 10703: 'val_loss' reached 0.24007 (best 0.24007), saving model to 'C:\\Users\\ADMIN\\OneDrive - Hanoi University of Mining and Geology\\Documents\\NCKH\\TFT-GreenPower-Forecasting\\checkpoint\\moirai2_v1_epoch=76_val_loss=0.2401.ckpt' as top 1


Epoch 77, global step 10842: 'val_loss' was not in top 1


Metric val_loss improved by 0.000 >= min_delta = 0.0. New best score: 0.240
Epoch 78, global step 10981: 'val_loss' reached 0.23978 (best 0.23978), saving model to 'C:\\Users\\ADMIN\\OneDrive - Hanoi University of Mining and Geology\\Documents\\NCKH\\TFT-GreenPower-Forecasting\\checkpoint\\moirai2_v1_epoch=78_val_loss=0.2398.ckpt' as top 1


Epoch 79, global step 11120: 'val_loss' was not in top 1
`Trainer.fit` stopped: `max_epochs=80` reached.


Training done
Best val_loss: 0.23978014290332794
Best ckpt: C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting\checkpoint\moirai2_v1_epoch=78_val_loss=0.2398.ckpt


In [13]:
BEST_CKPT = CKPT_DIR / 'moirai2_v1_best.ckpt'
LATEST_CKPT = CKPT_DIR / 'moirai2_v1_latest.ckpt'
PRETRAIN_WEIGHTS = CKPT_DIR / 'moirai2_v1_pretrain_weights.pt'
CONFIG_PATH = CKPT_DIR / 'moirai2_v1_config.json'

if checkpoint_callback.best_model_path and os.path.exists(checkpoint_callback.best_model_path):
    import shutil
    shutil.copy2(checkpoint_callback.best_model_path, BEST_CKPT)

if checkpoint_callback.last_model_path and os.path.exists(checkpoint_callback.last_model_path):
    import shutil
    shutil.copy2(checkpoint_callback.last_model_path, LATEST_CKPT)

state_obj = {
    'model_state_dict': lit_model.model.state_dict(),
    'feature_cols': feature_cols,
    'target_col': target_col,
    'cfg': CFG
}
torch.save(state_obj, PRETRAIN_WEIGHTS)

config = {
    'version': 'moirai2_v1',
    'best_ckpt': str(BEST_CKPT),
    'latest_ckpt': str(LATEST_CKPT),
    'pretrain_weights': str(PRETRAIN_WEIGHTS),
    'best_val_loss': float(checkpoint_callback.best_model_score) if checkpoint_callback.best_model_score is not None else None,
    'context_length': CFG['context_length'],
    'prediction_length': CFG['prediction_length'],
    'feature_cols': feature_cols,
    'target_col': target_col,
    'group_cols': group_cols,
    'quantiles': CFG['quantiles'],
    'd_model': CFG['d_model'],
    'nhead': CFG['nhead'],
    'num_layers': CFG['num_layers'],
    'dropout': CFG['dropout']
}

with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f'Best ckpt: {BEST_CKPT}')
print(f'Latest ckpt: {LATEST_CKPT}')
print(f'Pretrain weights: {PRETRAIN_WEIGHTS}')
print(f'Config: {CONFIG_PATH}')

Best ckpt: checkpoint\moirai2_v1_best.ckpt
Latest ckpt: checkpoint\moirai2_v1_latest.ckpt
Pretrain weights: checkpoint\moirai2_v1_pretrain_weights.pt
Config: checkpoint\moirai2_v1_config.json


In [14]:
best_model = LitMoirai2.load_from_checkpoint(str(BEST_CKPT), cfg=CFG, input_dim=input_dim)
best_model.eval()
best_model.to(DEVICE)

all_true = []
all_pred = []
median_idx = CFG['quantiles'].index(0.5) if 0.5 in CFG['quantiles'] else len(CFG['quantiles']) // 2

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        pred_q = best_model(x)
        pred = pred_q[:, :, median_idx]
        all_true.append(y)
        all_pred.append(pred)

y_true = torch.cat(all_true, dim=0)
y_pred = torch.cat(all_pred, dim=0)
mae, rmse, smape, wape, r2 = compute_metrics(y_true, y_pred)

print('Validation metrics (Moirai-2 pretrain):')
print(f'  MAE   : {mae:.4f} TWh')
print(f'  RMSE  : {rmse:.4f} TWh')
print(f'  R2    : {r2:.4f}')
print(f'  SMAPE : {smape:.2f}%')
print(f'  WAPE  : {wape:.2f}%')

Validation metrics (Moirai-2 pretrain):
  MAE   : 0.6415 TWh
  RMSE  : 1.9417 TWh
  R2    : 0.7526
  SMAPE : 45.28%
  WAPE  : 27.43%


c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightning\fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


In [15]:
print('=' * 70)
print('MOIRAI-2 PRETRAIN SUMMARY')
print('=' * 70)
print(f'Best val_loss : {checkpoint_callback.best_model_score}')
print(f'Best checkpoint: {BEST_CKPT}')
print(f'Config        : {CONFIG_PATH}')
print(f'Validation R2 : {r2:.4f}')
print('Next: run transfer_learning_moirai2_v1.ipynb for Vietnam fine-tuning')
print('=' * 70)

MOIRAI-2 PRETRAIN SUMMARY
Best val_loss : 0.23978014290332794
Best checkpoint: checkpoint\moirai2_v1_best.ckpt
Config        : checkpoint\moirai2_v1_config.json
Validation R2 : 0.7526
Next: run transfer_learning_moirai2_v1.ipynb for Vietnam fine-tuning
